# Лекция: Компьютерное зрение. 
## Адаптеры в диффузионных моделях & SOTA модели. Часть 2.
**Преподаватель**: Шевченко Елена, Т-Банк

**Аудитория**: ФКН ВШЭ, МФТИ

### Содержание
1. ~~Адаптеры~~
2. Recap: диифузия VS flow matching
3. SOTA models
   - Общие принципы
   - Flux 1
   - Flux-Kontext
   - Flux 2
   - Qwen-Image
5. Model distillation

# II. Recap: классическая диффузия VS Flow Matching

![flow_diffusion](assets/flow_diffusion.png)
### 1. Классическая диффузия

### 1.1 Идея

Мы хотим научиться **генерировать** изображения — то есть сэмплировать из распределения $p(x_0)$.

Напрямую смоделировать $p(x_0)$ слишком сложно.

Диффузионный подход решает это в два шага:

```
Прямой процесс:    x_0 ──► x_T        мы сами зашумляем данные
                                        до чистого гауссова шума

Обратный процесс:  x_T ──► x_0        учимся обращать зашумление
                                        → генерация нового изображения
```

Ключевая идея: зашумить просто, а научившись обращать зашумление — получим генератор.

---

### 1.2 Прямой процесс: проектируем SDE

Прямой процесс — это **SDE, которое мы сами выбираем**:

$$dx = f(x, t)\, dt + g(t)\, dW_t$$

Мы не угадываем $f$ и $g$ — мы их **проектируем**.

Конкретный выбор (DDPM):

$$f(x, t) = -\frac{1}{2}\beta(t)\, x, \qquad g(t) = \sqrt{\beta(t)}$$

где $\beta(t)$ — **noise schedule** — функция, которую мы задаём руками:

```
β(t) маленький при t≈0  →  почти не зашумляем
β(t) большой при t≈T   →  сильно зашумляем
```

Типичные варианты: линейный schedule, косинусный schedule.

---

### 1.3 Решение прямого SDE

Выбранное SDE имеет **аналитическое решение** — можно сразу записать $x_t$ для любого $t$:

$$x_t = \alpha_t x_0 + \sigma_t \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

где $\alpha_t$ и $\sigma_t$ аналитически выражаются через $\beta(t)$:

$$\alpha_t = e^{-\frac{1}{2}\int_0^t \beta(s)\,ds}, \qquad \sigma_t = \sqrt{1 - \alpha_t^2}$$

Это важно для обучения:

```
хотим зашумить x_0 до уровня t
→ не нужно делать t последовательных шагов
→ можно сразу сэмплировать x_t за одну операцию
→ это сильно ускоряет обучение
```

При $t \to T$: $\alpha_t \to 0$, $\sigma_t \to 1$ — изображение превращается в чистый шум $\mathcal{N}(0, I)$.

---

### 1.4 Обратный процесс: теорема Андерсона

Мы хотим идти в обратную сторону: $x_T \to x_0$.

**Теорема Андерсона (1982):** если прямой процесс — SDE с коэффициентами $f$ и $g$, то обратный процесс **тоже является SDE**:

$$dx = \Big[\underbrace{f(x,t)}_{\text{известно}} - \underbrace{g(t)^2}_{\text{известно}} \cdot \underbrace{\nabla_x \log p_t(x)}_{\text{неизвестно}}\Big]\, dt + g(t)\, d\bar{W}_t$$

Здесь:

- $f(x,t)$ и $g(t)$ — **те же самые**, что мы выбрали в 1.2, они известны
- $d\bar{W}_t$ — броуновский процесс, идущий в **обратном времени**
- $\nabla_x \log p_t(x)$ — **score function**, которую мы не знаем

---

### 1.5 Score function: почему она недоступна

$$\nabla_x \log p_t(x_t)$$

Это градиент логарифма плотности распределения зашумлённых изображений.

Чтобы её вычислить, нужно знать $p_t(x_t)$:

$$p_t(x_t) = \int p(x_0) \cdot p(x_t \mid x_0)\, dx_0$$

Это интеграл по **всем возможным изображениям** — аналитически неберётся.

Интуиция score function:

```
x_t — зашумлённое изображение

∇_x log p_t(x_t) указывает направление,
в котором плотность реальных изображений
растёт быстрее всего

то есть: "иди туда, где больше реальных данных"

шум ●  ← ← ← ← ← ← ←  ● реальное изображение
        score указывает сюда
```

---

### 1.6 Аппроксимация нейросетью

Раз score function недоступна аналитически — **обучаем нейросеть**:

$$s_\theta(x_t, t) \approx \nabla_x \log p_t(x_t)$$

Обучать score function напрямую неудобно. Но можно показать (score matching), что это **эквивалентно** предсказанию шума:

$$s_\theta(x_t, t) \approx -\frac{\epsilon_\theta(x_t, t)}{\sigma_t}$$

Поэтому на практике обучают сеть предсказывать $\epsilon$:

$$\mathcal{L}_{\text{diffusion}} = \mathbb{E}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

---

### 1.7 Откуда берётся стохастичность

Посмотрим на обратный процесс:

$$dx = \Big[f(x,t) - g(t)^2 \cdot \nabla_x \log p_t(x)\Big]\, dt + \underbrace{g(t)\, d\bar{W}_t}_{\text{зачем это?}}$$

Возникает вопрос: мы же хотим **убирать** шум — зачем добавлять новый?

Ответ: это математическое следствие выбора стохастического прямого процесса.

```
Прямой процесс добавляет шум
→ часть информации об x_0 теряется безвозвратно
→ обратный процесс не может быть детерминированным
→ нужно "угадывать" потерянную информацию
→ шум в обратном процессе — это и есть это угадывание
```

Формально: если прямой процесс — SDE, то по теореме Андерсона обратный **тоже SDE**.

---

### 1.8 Проблемы стохастичности

#### Проблема 1: накопление ошибок

Каждый шаг обратного процесса содержит два источника ошибок:

```
ошибка сети:   s_θ ≈ ∇ log p_t   неточная аппроксимация
ошибка шума:   случайный ΔW       случайное отклонение от траектории
```

Эти ошибки накапливаются


#### Проблема 2: искривлённые траектории

Диффузионный процесс зашумляет данные нелинейно — траектории искривлены:

```
x_0 ─────────────────────── x_T
      кривая траектория
```

Для точного интегрирования искривлённой траектории нужны **маленькие шаги**.

#### Проблема 3: много шагов

Стохастическое интегрирование требует малого $\Delta t$ для стабильности:

```
большой Δt  →  нестабильная траектория  →  артефакты
маленький Δt  →  много шагов  →  медленно
```

Типично: **20–100 шагов**.

#### Итог

```
выбор стохастического SDE
    ↓
стохастический обратный процесс
    ↓
накопление ошибок + искривлённые траектории
    ↓
много шагов → медленная генерация
```

Все эти проблемы — **не случайность**. Они прямое следствие выбора стохастического прямого процесса.

---

### 1.9 Итоговая картина

Диффузия стохастична не потому что так надо, а потому что мы выбрали стохастический прямой процесс. Это порождает накопление ошибок, искривлённые траектории и необходимость в большом числе шагов.

Возникает вопрос: можно ли построить прямой процесс так, чтобы обратный был **детерминированным**?


---

## 2. Flow Matching

### 2.1 Ключевой сдвиг

Диффузия устроена так:

```
1. Проектируем прямой процесс (SDE)
2. Хотим обратить его
3. Обращение требует score function ∇ log p_t(x)
4. Score function аналитически недоступна
5. Аппроксимируем её нейросетью
```

Проблемы диффузии возникают из-за того, что прямой процесс стохастический. Это делает обратный процесс тоже стохастическим, траектории искривлёнными, а score function — сложной целью для обучения.

Flow Matching задаёт другой вопрос:

> А что если выбрать прямой процесс так, чтобы скорость обратного движения была **аналитически вычислима**? Тогда сети не нужно аппроксимировать сложную score function — она просто предсказывает известную скорость.
> 

Flow Matching устроен иначе:

```
1. Проектируем прямой процесс (линейная интерполяция)
2. Вычисляем скорость вдоль траекторий аналитически
3. Обучаем сеть предсказывать эту скорость
4. Генерация — интегрируем ODE в обратную сторону
```

Ключевое отличие: в диффузии сеть вынуждена **восстанавливать** то, что аналитически недоступно. В Flow Matching сеть **аппроксимирует** то, что мы сами вычислили.

---

### 2.2 Идея: ODE вместо SDE

Вместо SDE рассмотрим **ODE**:

$$\frac{dx}{dt} = v_\theta(x, t)$$

Здесь нет $dW_t$ — процесс полностью **детерминированный**.

Модель предсказывает **вектор скорости** (velocity field) — направление и величину движения в каждой точке пространства в момент времени $t$.

Сразу видно преимущество:

```
нет стохастичности
→ нет накопления ошибок от шума
→ можно делать крупные шаги
→ меньше шагов → быстрее генерация
```

Но возникает вопрос: **как построить** такое поле скоростей?

---

### 2.3 Построение траектории

Flow Matching отвечает на этот вопрос напрямую.

Зададим **линейную интерполяцию** между данными и шумом:

$$x_t = (1 - t)\, x_0 + t\, z, \qquad z \sim \mathcal{N}(0, I), \quad t \in [0, 1]$$

```
t = 0:  x_t = x_0        чистое изображение
t = 0.5:  x_t = смесь    промежуточное состояние
t = 1:  x_t = z          чистый шум
```

Это и есть прямой процесс — **линейный и детерминированный** (для фиксированной пары $x_0, z$).

Такой выбор называют **Rectified Flow** — это Flow Matching с линейной интерполяцией

---

### 2.4 Целевая скорость

Вычислим скорость вдоль построенной траектории:

$$v^*(x_t, t) = \frac{dx_t}{dt} = \frac{d}{dt}\Big[(1-t)\,x_0 + t\,z\Big] = z - x_0$$

Скорость $z - x_0$ **константна** вдоль одной условной траектории** при фиксированных $(x_0, z)$.

Геометрически:

```
Диффузия:

x_0 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~ z
    кривая, скорость меняется

Flow Matching:

x_0 ──────────────────────────── z
    прямая, скорость = const = z - x_0
```

Прямые траектории — это и есть **Rectified Flow**.

---

### 2.5 Функция потерь

Обучаем сеть предсказывать целевую скорость:

$$\mathcal{L}_{\text{flow}} = \mathbb{E}_{t,\, x_0,\, z}\left[\|v_\theta(x_t, t) - (z - x_0)\|^2\right]$$

Сравним с диффузией:

|  | Цель предсказания | Сложность цели |
| --- | --- | --- |
| Diffusion | шум $\epsilon$ | зависит от $\sigma_t$, нелинейно |
| Flow Matching | скорость $z - x_0$ | **константа** вдоль траектории |

Предсказывать константу проще — градиенты стабильнее.

---

### 2.6 Генерация (sampling)

Генерация — это интегрирование ODE **в обратную сторону** ($t: 1 \to 0$):

$$x_{t - \Delta t} = x_t - v_\theta(x_t, t)\, \Delta t$$

Простейший метод — Euler:

```
начинаем с z ~ N(0, I)
делаем несколько шагов ODE
получаем x_0 ≈ реальное изображение
```

Благодаря прямым траекториям достаточно **4–16 шагов**.

---

### 2.7 Почему прямые траектории так важны

При численном интегрировании ODE ошибка на каждом шаге зависит от **кривизны траектории**:

```
высокая кривизна  →  большая ошибка при крупном шаге
                  →  нужны мелкие шаги
                  →  много итераций

низкая кривизна   →  малая ошибка при крупном шаге
(прямая линия)    →  можно делать крупные шаги
                  →  мало итераций
```

Линейная интерполяция даёт минимально возможную кривизну — прямую линию.

---

### 2.8 Сравнение диффузии и Flow Matching

| Критерий | Диффузия | Flow Matching |
| --- | --- | --- |
| Тип процесса | SDE (стохастический) | ODE (детерминированный) |
| Траектории | кривые, нелинейные | прямые, линейные |
| Цель предсказания | шум $\epsilon$ | скорость $v = z - x_0$ |
| Число шагов | 20–100 | 4–16 |
| Накопление ошибок | есть (шум + сеть) | только сеть |
| Воспроизводимость | нет | да |
| Стабильность обучения | ниже | выше |

---

### 2.9 Важное замечание

Flow Matching меняет:

- **прямой процесс** — линейная интерполяция вместо SDE
- **функцию потерь** — предсказание скорости вместо шума
- **генерацию** — ODE вместо SDE

Но **не меняет**:

- архитектуру модели
- латентное пространство
- conditioning механизмы

Именно поэтому flow matching легко интегрируется в существующие архитектуры — включая FLUX.

---

### 2.10 Главная мысль

> Диффузия вынуждена использовать стохастический обратный процесс, потому что прямой процесс — стохастический.
> 
> 
> Flow Matching меняет сам выбор: строит **линейный детерминированный** прямой процесс, для которого обратный процесс — простое ODE с константной скоростью.
> 
> Это убирает стохастичность, выпрямляет траектории, стабилизирует обучение и делает генерацию быстрее.
>

# II. Современные архитектуры

## 3.1 Эволюция архитектур

### Первое поколение: U-Net

![dif_unet](assets/diffusion_unet.png)

Исторически первые диффузионные модели (DDPM, Stable Diffusion 1.x) использовали **U-Net** :

```
вход → encoder → bottleneck → decoder → выход
         ↑________________________↑
              skip connections
```

U-Net изначально разработан для сегментации изображений. Его адаптировали для диффузии:

- добавили attention слои в bottleneck
- добавили conditioning через cross-attention
- добавили timestep embedding

**Проблемы U-Net для больших моделей:**

- архитектура заточена под изображения фиксированного размера
- плохо масштабируется с числом параметров
- cross-attention между текстом и изображением — отдельный механизм, не интегрированный органично

---

### Второе поколение: Diffusion Transformer (DiT)

![dit](assets/dit.png)

В 2022–2023 годах появилась идея: заменить U-Net на **чистый трансформер** .

Ключевое наблюдение:

> Изображение можно разбить на патчи и обрабатывать как последовательность токенов — так же, как текст в языковых моделях.
> 

```
изображение → патчи → токены → transformer → токены → изображение
```

**Что это даёт:**

- масштабирование по законам scaling laws (как в LLM)
- единый механизм attention для всех токенов
- гибкость к разрешению и соотношению сторон

---

### Третье поколение: Multimodal DiT (MM-DiT)

![mm-dit](assets/mm-dit.png)

Следующий шаг — обрабатывать текст и изображение **в одном трансформере**, но с учётом их различной природы


---

## 3.2 Общие принципы SOTA архитектур

Современные топовые генеративные модели объединяют четыре принципа.

---

### Принцип 1: Flow Matching вместо диффузии

Это самый фундаментальный сдвиг последних лет.

Исторически все модели использовали диффузионный objective — предсказание шума $\epsilon$. Но как мы разобрали в части 2, это порождает стохастичность, искривлённые траектории и необходимость в большом числе шагов.

Современные SOTA модели переходят на **Rectified Flow**:

```
предсказываем не шум ε
а скорость v = z - x₀
```

Это меняет только **objective обучения** — не архитектуру. Поэтому переход относительно безболезненный.

**Кто использует:**

| Модель | Objective |
| --- | --- |
| DALL-E 2, SD 1.x | диффузия (предсказание $\epsilon$) |
| SD 3, FLUX, Sora | Rectified Flow |

---

### Принцип 2: трансформеры вместо U-Net

Это требует отдельного объяснения — почему вообще трансформеры?

**Проблема U-Net:**

U-Net — свёрточная архитектура. Свёртки работают **локально** — каждый нейрон видит только небольшую окрестность.

```
свёртка 3×3: видит только соседей
→ глобальный контекст накапливается медленно
→ нужна глубокая сеть чтобы "увидеть" всё изображение
```

Кроме того, U-Net плохо масштабируется:

```
больше параметров → нестабильное обучение
нестандартное разрешение → нужна перестройка архитектуры
текст + изображение → awkward cross-attention
```

**Почему трансформер лучше:**

U-Net использует attention частично, как дополнение к свёрточной основе. DiT строит всю архитектуру на attention — это даёт лучшее масштабирование и более однородную обработку всех токенов на каждом слое.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d}}\right)V$$

Это означает:

```
любой патч изображения может напрямую
"общаться" с любым другим патчем
→ глобальный контекст с первого слоя
→ не нужна глубина для распространения информации
```

**Масштабирование:**

Трансформеры подчиняются **scaling laws** — качество предсказуемо растёт с числом параметров и данных . Это хорошо изучено на LLM.

```
больше параметров → предсказуемо лучше
больше данных     → предсказуемо лучше
```

U-Net такими свойствами не обладает.

**Унификация модальностей:**

В трансформере текст и изображение — просто **последовательности токенов**. Их можно обрабатывать в одной архитектуре без специальных механизмов.

То есть латент разбивается на патчи размером $p \times p$:

$$\text{латент } H \times W \times C \;\to\; \text{токены } \frac{HW}{p^2} \times d$$

Каждый патч линейно проецируется в вектор размерности $d$ — это и есть токен.

```
U-Net:        текст → cross-attention → изображение
                      отдельный механизм

Transformer:  [текст токены | изображение токены]
                      единый self-attention
```
---

### Принцип 3 — Conditioning через AdaLN и joint attention

Модель должна получать несколько типов сигналов: timestep $t$, текст $c$, guidance scale.

**Скалярные сигналы: AdaLN**

AdaLN — устоявшийся инструмент, появился ещё в «Diffusion Beat GANs» (2021). Идея: вместо того чтобы складывать conditioning с токенами на входе, управлять **параметрами нормализации в каждом слое**:

$$y = \gamma(c) \cdot \text{LayerNorm}(x) + \beta(c)$$

Используется во всех современных моделях для скалярных сигналов — timestep и guidance.

**Текст: от cross-attention к joint attention**

Текст участвует в **joint attention** наравне с визуальными токенами: 

$\text{tokens} = [\underbrace{t_1, t_2, \ldots, t_n}_{\text{текстовые токены}},\ \underbrace{v_1, v_2, \ldots, v_m}_{\text{визуальные токены}}]$

---

### Принцип 4 — Rotary Positional Encoding (RoPE)

Трансформер обрабатывает токены без понятия о порядке — self-attention симметричен. Нужен способ сообщить модели, где находится каждый токен.

Классическое решение — добавить позиционный вектор к токену:

$$\text{input}_i = \text{token}_i + \text{pos\_emb}_i$$

**Проблема:** позиции фиксированы при обучении. При другом разрешении модель встречает незнакомые позиции → плохая экстраполяция.

RoPE решает это иначе. Вместо того чтобы добавлять позицию к токену, RoPE **вращает** векторы запросов $Q$ и ключей $K$ перед вычислением attention. Угол вращения зависит от позиции токена.

Ключевое свойство: скалярное произведение после вращения зависит только от **относительного расстояния** между токенами:

$$q_m^T k_n = f(m - n)$$

```
позиции m=5,   n=3   → зависит от 5-3=2
позиции m=105, n=103 → зависит от 105-103=2

результат одинаковый ✓
```

Для изображений RoPE расширяется на **2D** — вектор делится на две половины, каждая вращается по своей оси:

$$\text{вектор размерности } d = \underbrace{d/2}_{\text{ось X (столбец)}} + \underbrace{d/2}_{\text{ось Y (строка)}}$$

Это и даёт гибкость к разрешению:

```
обучение:  1024×1024 → токены в сетке [0..127]²
генерация: 1024×512  → токены в сетке [0..127]×[0..63]

модель не видела такой сетки
но RoPE кодирует относительные расстояния
→ "этот токен на 3 строки ниже соседнего"
→ работает при любом разрешении ✓
```

---

### Итог: четыре принципа

| Принцип | Что решает |
| --- | --- |
| Flow Matching | стохастичность, медленная генерация |
| Трансформер вместо U-Net | масштабирование, глобальный контекст, унификация модальностей |
| AdaLN + Joint-atn conditioning | эффективное внедрение conditioning |
| RoPE | работа с произвольными разрешениями |




## 3.3 Семейство FLUX: FLUX 1.x

![flux](assets/flux1.png)

### Контекст появления

FLUX — семейство моделей от **Black Forest Labs**, основанной в августе 2024 года выходцами из Stability AI

FLUX.1 при выходе превзошёл по качеству Midjourney V6, DALL-E 3 и Stable Diffusion 3 Ultra по метрике ELO-score (human preference ranking)

> Примечание: официальная техническая документация по архитектуре и обучению не была опубликована. Архитектура восстановлена исследователями по открытому inference коду

**Связь с SD3**

```
SD3 использует:
→ MM-DiT архитектуру (multimodal diffusion transformer)
→ совместную обработку текста и изображения

FLUX.1 использует:
→ ту же базовую идею MM-DiT
→ но развитую дальше: DoubleStream + SingleStream
→ + Rectified Flow вместо классической диффузии
→ + 12B параметров против 8B у SD3
```

### Общий взгляд

**FLUX имеет три ключевых компонента:**

```
1. VAE          — кодирует/декодирует изображение в латент
2. Text encoder — кодирует текстовый промпт
3. Transformer  — предсказывает поле скоростей v_θ(x_t, t)
```

Objective — **Rectified Flow**:

$$\mathcal{L} = \mathbb{E}\left[\|v_\theta(x_t, t) - (z - x_0)\|^2\right]$$


FLUX.1 использует **два** текстовых энкодера одновременно:

| Энкодер | Роль |
| --- | --- |
| **T5-XXL** | длинные семантические описания, детали промпта |
| **CLIP** | визуально-семантическое выравнивание |

Почему два?

```
CLIP обучен на пары (изображение, текст)
→ хорошо понимает визуальные концепции
→ но ограничен короткими описаниями

T5 обучен на длинных текстах
→ хорошо понимает сложные промпты
→ но не видел изображений
```

**Латентное пространство и токенизация**

```
изображение 1024×1024
      ↓ VAE encoder
латент 128×128×16
      ↓ patchify (2×2 патчи)
токены ~4096 × d
```

Итого: **около 4096 токенов** на одно изображение.

---

## Архитектура Flux 1

### Modulation: как conditioning управляет каждым блоком

Модель должна вести себя по-разному на разных шагах генерации. На раннем шаге ($t \approx 1$, почти чистый шум) — формировать грубую структуру. На позднем ($t \approx 0$, почти готовое изображение) — прорабатывать детали.

Наивное решение — сложить timestep embedding с токенами на входе:

```
x ← x + embedding(t)
```

Проблема: влияет только на входе, затухает с глубиной.

**Modulation** решает это иначе. Conditioning вектор $c$ (timestep + guidance) проходит через линейный слой и разбивается на три части:

```
c → Linear → [shift, scale, gate]
```

**shift и scale** модулируют нормализацию в каждом блоке (AdaLN):

$$\text{modulate}(x, c) = (1 + \text{scale}(c)) \cdot \text{RMSNorm}(x) + \text{shift}(c)$$

**gate** контролирует насколько сильно выход attention или MLP меняет токены:

$$x \leftarrow x + \text{gate}(c) \cdot \text{Attention}(\ldots)$$

```
gate ≈ 0  →  выход attention почти не меняет токены
gate ≈ 1  →  выход attention полностью добавляется
```

Полная схема внутри блока:

```
conditioning c
      ↓
   Linear
      ↓
[shift, scale, gate_attn, shift2, scale2, gate_mlp]

x → (1 + scale)·RMSNorm(x) + shift    ← modulate перед attention
          ↓
      Attention
          ↓
x ← x + gate_attn · Attention(...)     ← gate после attention
          ↓
x → (1 + scale2)·RMSNorm(x) + shift2  ← modulate перед MLP
          ↓
         MLP
          ↓
x ← x + gate_mlp · MLP(...)            ← gate после MLP
```

На пальцах:

```
ранний шаг (t ≈ 1):
→ gate большой → сильные изменения → грубая композиция

поздний шаг (t ≈ 0):
→ gate маленький → тонкие изменения → детали
```

---

### QK Normalization через RMSNorm

Перед вычислением attention применяется **RMSNorm** к векторам $Q$ и $K$:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2 + \epsilon}} \cdot \gamma$$

Зачем: joint attention по длинной последовательности (~4096 токенов) может давать очень большие значения $QK^T$:

```
большие QK^T
→ softmax насыщается
→ градиенты почти нулевые
→ нестабильное обучение
```

RMSNorm держит нормы $Q$ и $K$ под контролем до вычисления attention. Это особенно важно в FLUX — длинные последовательности, большая модель, joint attention объединяет токены двух модальностей с разной статистикой.

---

### DoubleStream Blocks — специализация (19 блоков)

Текстовые и визуальные токены обрабатываются **раздельными весами**, но участвуют в **одном joint attention**.

```
текстовые токены  → Wq_text,  Wk_text,  Wv_text  → Q_t, K_t, V_t
визуальные токены → Wq_image, Wk_image, Wv_image → Q_v, K_v, V_v

RMSNorm(Q_t), RMSNorm(K_t)     ← QK norm
RMSNorm(Q_v), RMSNorm(K_v)

конкатенация:
Q = [Q_t, Q_v]
K = [K_t, K_v]
V = [V_t, V_v]
        ↓
Attention(Q, K, V)  ← один проход, все модальности видят друг друга
        ↓
разделяем → обновлённые text tokens, image tokens
        ↓
MLP_text       MLP_image    ← раздельные MLP
```

Modulation тоже **раздельная** — у каждого потока свои shift, scale, gate:

```
c → Linear_text  → [shift_t, scale_t, gate_attn_t, shift2_t, scale2_t, gate_mlp_t]
c → Linear_image → [shift_v, scale_v, gate_attn_v, shift2_v, scale2_v, gate_mlp_v]
```

Это позволяет conditioning по-разному влиять на текстовые и визуальные токены.


Полная схема одного DoubleStream блока:

```
         conditioning c
               ↓
    ┌──────────────────────────┐
    │ Linear_text  Linear_image│
    └──────┬───────────┬───────┘
           ↓           ↓
    [shift,scale,  [shift,scale,
      gate]_text    gate]_image
           ↓           ↓
    ┌──────┴───────────┴───────┐
    │   modulate (раздельный)  │
    └──────┬───────────┬───────┘
           ↓           ↓
    Wq,Wk,Wv_text  Wq,Wk,Wv_image
           ↓           ↓
    Q_t,K_t,V_t   Q_v,K_v,V_v
           ↓           ↓
    RMSNorm(Q_t)  RMSNorm(Q_v)
    RMSNorm(K_t)  RMSNorm(K_v)
           ↓           ↓
         concat Q, K, V
               ↓
         Attention(Q,K,V)
               ↓
         разделяем выход
           ↓           ↓
    gate_t · out_t  gate_v · out_v
           ↓           ↓
    x_t += ...     x_v += ...
           ↓           ↓
    MLP_text       MLP_image
           ↓           ↓
    gate_mlp_t·   gate_mlp_v·
      MLP_t(...)    MLP_v(...)
           ↓           ↓
    x_t += ...     x_v += ...
```

---

### SingleStream Blocks — интеграция (38 блоков)

К этой стадии токены обоих потоков уже достаточно согласованы. Их **конкатенируют** и обрабатывают как единую последовательность с **общими весами**:

```
[text tokens, image tokens]
          ↓ concat
    единая последовательность
          ↓
    одни веса Wq, Wk, Wv
          ↓
RMSNorm(Q), RMSNorm(K)         ← QK norm
          ↓
    Attention(Q_all, K_all, V_all)
          ↓
        один MLP
```

Modulation тоже **общая** — один набор shift, scale, gate для всей последовательности:

```
c → Linear → [shift, scale, gate_attn, shift2, scale2, gate_mlp]
                    ↓
             вся последовательность
```

**На пальцах:** SingleStream — это как два студента, которые теперь думают вместе, пишут одной ручкой, получают одни указания от преподавателя и не различают где чья мысль.


#### Итоговое сравнение

|  | DoubleStream | SingleStream |
| --- | --- | --- |
| Блоков | 19 | 38 |
| Веса проекций Q,K,V | раздельные | общие |
| MLP | раздельные | общий |
| Modulation | раздельная для каждого потока | общая |
| QK Norm | да | да |
| Joint attention | да | да |
| Роль | специализация | интеграция |
| Аналогия | думают рядом, но по-своему | думают вместе |

---

### Итоговая схема

```
text tokens + image tokens
          ↓
┌─────────────────────────┐
│   19× DoubleStream      │
│   раздельные веса       │
│   раздельная modulation │
│   joint attention       │
└──────────┬──────────────┘
           │ concat
┌──────────▼──────────────┐
│   38× SingleStream      │
│   общие веса            │
│   общая modulation      │
│   full self-attention   │
└──────────┬──────────────┘
           ↓
      velocity field
      v_θ(x_t, t)
```

> DoubleStream и SingleStream решают одну задачу в два этапа: сначала дать каждой модальности выработать своё представление с раздельными весами и раздельной modulation, потом объединить их в единое совместное представление. Modulation при этом обеспечивает динамическое управление силой изменений на каждом шаге генерации — через gate механизм conditioning вектор говорит каждому слою насколько «сильно» работать прямо сейчас.


### Classifier-Free Guidance в FLUX.1

В классическом CFG нужно два прохода модели:

$$\hat{v} = v_\theta(x_t, \varnothing) + w\left(v_\theta(x_t, c) - v_\theta(x_t, \varnothing)\right)$$

FLUX.1 [dev] использует **guidance distillation** — guidance передаётся как embedding, один проход

---

### Варианты FLUX.1

| Вариант | Параметры | Шаги | Особенность | Лицензия |
| --- | --- | --- | --- | --- |
| FLUX.1 [pro] | 12B | — | максимальное качество | API only |
| FLUX.1 [dev] | 12B | 20–50 | guidance distillation | некоммерческая |
| FLUX.1 [schnell] | 12B | 1–4 | latent adversarial distillation | Apache 2.0 |

FLUX.1 [schnell] обучен с применением **latent adversarial diffusion distillation (LADD)** — это позволяет генерировать за 1–4 шага.

---

### Почему FLUX работает

Не из-за одной идеи, а из-за **согласованного набора решений**:

| Компонент | Роль |
| --- | --- |
| Rectified Flow objective | прямые траектории, быстрая генерация |
| DoubleStream → SingleStream | стабильное обучение + глубокая интеграция |
| Dual text encoders | семантика + визуальное выравнивание |
| AdaLN conditioning | гибкое управление на каждом слое |
| RoPE | работа с длинными последовательностями и произвольными разрешениями |
| 12B параметров | достаточная ёмкость модели |


## 3.4 FLUX.1 Kontext

![flux-kontext](assets/flux-kontext.png)

### Что такое FLUX Kontext и зачем он нужен

FLUX.1 (базовая модель) умеет только одно: **генерировать изображение из текста**. Каждый раз — с нуля, из чистого шума.

Это создаёт фундаментальную проблему для редактирования:

```
классический подход к редактированию:
1. берём исходное изображение
2. добавляем шум (inversion)
3. запускаем диффузию с новым промптом
4. получаем новое изображение

проблемы:
→ изображение меняется глобально, не только нужная область
→ стиль и композиция нестабильны
```

FLUX Kontext решает это иначе: модель обучена **понимать референсное изображение как часть контекста** и вносить только запрошенные изменения .

---

### Ключевая архитектурная идея: sequence concatenation

Главное архитектурное решение Kontext — простое и элегантное :

```
базовый FLUX.1:
вход → [text tokens] + [noisy image tokens]
             ↓
        transformer blocks
             ↓
        velocity field v_θ

FLUX Kontext:
вход → [text tokens] + [reference image tokens] + [noisy target tokens]
                          ↑
                    новое: референсное
                    изображение просто
                    конкатенируется в
                    ту же последовательность
             ↓
        те же transformer blocks (без изменений архитектуры!)
             ↓
        velocity field v_θ для target tokens
```

**На пальцах:** Kontext не меняет архитектуру трансформера. Он просто добавляет токены референсного изображения в ту же последовательность, которую уже умеет обрабатывать FLUX. Joint attention сам разберётся — теперь target токены могут "смотреть" на референс через механизм внимания.
```
Reference Latent [1,128,55,74] → patchify → [1, 4070, 128]
Noisy Latent    [1,128,55,74] → patchify → [1, 4070, 128]
                                    ↓
                               CONCATENATE
                                    ↓
                            [1, 8140, 128]
                                    ↓
                                  img_in
                                    ↓
                            [1, 8140, 4096]
```

---

### Как это работает технически

Референсное изображение проходит тот же путь, что и обычное изображение для генерации:

```
reference image
      ↓
   VAE encoder          ← тот же, что в базовом FLUX
      ↓
latent representation   ← непрерывное представление в латентном пространстве
      ↓
   patchify             ← разбивка на патчи, как в базовом FLUX
      ↓
reference tokens        ← готовы к конкатенации
```

Затем формируется общая последовательность:

```
[T5 text tokens | CLIP text tokens | reference tokens | noisy target tokens]
                                         ↑                    ↑
                               из референсного         то, что нужно
                               изображения             сгенерировать/отредактировать
```

Все токены участвуют в **одном joint attention** — DoubleStream и SingleStream блоки работают как обычно, только последовательность стала длиннее .

---

### 3D RoPE: новый способ позиционного кодирования

В базовом FLUX.1 используются **2D RoPE** — позиционные эмбеддинги кодируют только координаты `(x, y)` в пространстве изображения.

В Kontext нужно различать: где находится токен **и из какого изображения** он пришёл. Для этого вводятся **3D RoPE** с координатами `(image_id, x, y)` :

```
базовый FLUX: RoPE(x, y)
                  ↓
           модель знает где токен
           находится в изображении

FLUX Kontext: RoPE(image_id, x, y)
                  ↓
           модель знает: это токен из
           референсного изображения (id=0)
           или из целевого (id=1)
           И где именно он находится
```

```
пример:
reference image patch (100, 50)  → RoPE(0, 100, 50)
target image patch    (100, 50)  → RoPE(1, 100, 50)

без 3D RoPE: модель не может различить
             одинаково расположенные патчи
             из разных изображений

с 3D RoPE:   чётко различает источник
             каждого токена
```

Это позволяет attention правильно соотносить пространственные позиции референса и целевого изображения.

---

### Flow matching loss для редактирования

Обучающая цель Kontext — тот же flow matching, что и в базовом FLUX, но теперь модель учится **предсказывать velocity field для target токенов**, имея доступ к reference токенам:

**Базовый FLUX:**

$$x_t = (1-t) \cdot x_0 + t \cdot z$$

$$\mathcal{L} = \mathbb{E}\left[\|v_\theta(x_t, t) - (z - x_0)\|^2\right]$$

**Kontext:**

$$x_t^{\text{target}} = (1-t) \cdot x_0^{\text{target}} + t \cdot z^{\text{target}}$$

$$\mathcal{L} = \mathbb{E}\left[\|v_\theta([\text{ref},\ x_t^{\text{target}}],\ t) - (z^{\text{target}} - x_0^{\text{target}})\|^2\right]$$


> Важный момент: референсные токены не зашумляются. Модель всегда видит референс в чистом виде и учится изменять только target, сохраняя нужные характеристики.

---

### Итоговая схема FLUX Kontext

```
reference image          text prompt          noisy target
      ↓                      ↓                     ↓
   VAE enc             T5 + CLIP encoder        patchify
      ↓                      ↓                     ↓
ref patches           text tokens            noisy patches
      ↓                      ↓                     ↓
  patchify                   │                     │
      ↓                      │                     │
      └──────────────────────┴─────────────────────┘
                             ↓
              конкатенация в одну последовательность
         [text | ref patches | noisy target patches]
                             ↓
                    3D RoPE позиционирование
              (image_id, x, y) для каждого токена
                             ↓
              ┌──────────────────────────────┐
              │     19× DoubleStream blocks  │
              │  joint attention: все токены │
              │  видят друг друга            │
              └──────────────┬───────────────┘
                             ↓ concat
              ┌──────────────────────────────┐
              │     38× SingleStream blocks  │
              │  full self-attention         │
              └──────────────┬───────────────┘
                             ↓
              velocity field только для target tokens
                             ↓
                  денойзинг target image
                             ↓
                  VAE decoder → output image
```

---

### Что умеет Kontext благодаря этой архитектуре

Единая архитектура обрабатывает несколько типов задач без переключения между разными моделями :

| Задача | Что подаётся на вход | Что генерируется |
| --- | --- | --- |
| Text-to-image | только текст | новое изображение |
| Локальное редактирование | изображение + инструкция | то же изображение с изменённой областью |
| Глобальное редактирование | изображение + инструкция | трансформированное изображение |
| Character reference | персонаж + новый контекст | персонаж в новой сцене |
| Style reference | изображение + новый стиль | переstilized изображение |
| Text editing в изображении | изображение + новый текст | изображение с изменённым текстом |

---

### Версии моделей

Black Forest Labs выпустили три варианта :

```
FLUX.1 Kontext [pro]
→ быстрое итеративное редактирование
→ text + reference images как вход
→ до 10× быстрее предыдущих SOTA моделей
→ API-only

FLUX.1 Kontext [max]
→ максимальная производительность
→ улучшенное следование промптам
→ улучшенная типографика
→ API-only

FLUX.1 Kontext [dev]
→ открытые веса (open weights)
→ 12B параметров
→ guidance-distilled версия
→ для локального запуска и кастомизации
→ поддержка LoRA и ControlNet
```

---

### KontextBench: бенчмарк для оценки

Вместе с моделью авторы представили новый бенчмарк для оценки качества редактирования :

```
KontextBench:
→ 1026 пар (изображение, промпт-инструкция)
→ 5 категорий задач:
   1. local editing      — изменение конкретной области
   2. global editing     — изменение всего изображения
   3. character reference — сохранение персонажа
   4. style reference    — применение стиля
   5. text editing       — редактирование текста на изображении
→ оценивается:
   - single-turn quality   — качество одного редактирования
   - multi-turn consistency — стабильность при нескольких итерациях
```

---

> FLUX.1 Kontext — это не новая архитектура, а **умное расширение** уже описанных DoubleStream/SingleStream блоков. Главное изобретение: референсное изображение конкатенируется в ту же последовательность токенов через 3D RoPE, и существующий joint attention механизм сам учится извлекать нужную информацию из контекста, не изменяя структуру трансформера .
>

## 3.5 FLUX.2: второе поколение

![flux2](assets/flux2.png)

FLUX.2 вышел **25 ноября 2025 года** . Black Forest Labs описывают его как переход от «демо-инструмента» к production-grade системе — модель проектировалась под реальные рабочие задачи: рекламные кампании, каталоги товаров, инфографика, UI-макеты .

Важное отличие от FLUX.1: **это не отдельные модели для генерации и редактирования** — единая система, которая делает и то и другое .

---

### Что изменилось по сравнению с FLUX.1

```
FLUX.1 (август 2024):
→ 12B параметров
→ text encoders: T5 + CLIP
→ генерация и редактирование — разные модели
→ до 1 референсного изображения (Kontext)
→ максимум ~1MP

FLUX.2 (ноябрь 2025):
→ 32B параметров
→ text encoder: Mistral-3-3 (24B VLM)
→ генерация и редактирование — единая модель
→ до 10 референсных изображений
→ генерация и редактирование до 4MP
→ улучшенная типографика
→ JSON structured prompts
→ HEX color control
```

---

### Архитектура: что известно

FLUX.2 построен на **latent flow matching архитектуре** — комбинация rectified flow трансформера и vision-language модели Mistral-3 .


```
Flux2Pipeline использует:
→ transformer: Flux2Transformer2DModel   ← новый класс
→ vae: AutoencoderKLFlux2               ← новый VAE
→ text_encoder: Mistral-3               ← новый энкодер
```

Особенности :

```
→ SwiGLU вместо GELU в MLP блоках
→ fusion QKV проекций с FF входом в SingleStream
→ переобученный VAE
```

### Изменения в SingleStream блоке


**1. GELU → SwiGLU:**

```
FLUX.1 MLP:
x → W1·x → GELU(·) → W2·x

FLUX.2 MLP:
x → [W1; Wgate]·x → SwiGLU(W1·x, Wgate·x) → W2·x

где SwiGLU(x, gate) = x · SiLU(gate)
                            ↑
                    SiLU(x) = x · sigmoid(x)
```

**2. Fusion QKV + FF input:**

В FLUX.1 четыре отдельных матричных умножения:

```python
q = Wq(x)   # x читается из памяти
k = Wk(x)   # x читается снова
v = Wv(x)   # x читается снова
f = W1(x)   # x читается снова
```

В FLUX.2 — одно большое умножение:

```python
W_fused = Linear(d, 3*d + d_ff)   # одна большая матрица
out = W_fused(x)                   # x читается один раз
q, k, v, f = out.split([d, d, d, d_ff], dim=-1)
```

Математически результат идентичен — это инженерная оптимизация:

```
четыре умножения:
→ x читается из памяти 4 раза
→ четыре отдельных запуска операций на GPU

fusion:
→ x читается 1 раз
→ одна большая GEMM
→ GPU утилизируется эффективнее
→ особенно важно при 32B параметрах
```

---

### Новый текстовый энкодер: Mistral-3

В FLUX.1 текст кодировался связкой T5 + CLIP:

```
FLUX.1:
промпт → T5  (понимание длинного текста)
       → CLIP (семантическое выравнивание с изображением)
       → два типа text embeddings
```

В FLUX.2 оба заменены на **Mistral-3** — 24B vision-language модель :

```
FLUX.2:
промпт → Mistral-3 (24B VLM)
       → единые богатые text embeddings
       → stacked intermediate layers (не только последний слой)
```

Почему stacked intermediate layers важно:

```
FLUX.1 (T5):     берём выход последнего слоя
FLUX.2 (Mistral-3): берём и складываем выходы
                  нескольких промежуточных слоёв
                  → более богатые embeddings
                  → лучше захватывают структуру промпта
```

---

### Переобученный VAE: решение трилеммы

FLUX.2 переобучил латентное пространство **с нуля** .

Фундаментальное противоречие в VAE:

```
Learnability-Quality-Compression trilemma:

сильное сжатие → маленький латент → легко учить трансформер
                                   → плохое качество реконструкции

слабое сжатие  → большой латент   → хорошее качество
                                   → тяжело учить трансформер
```

FLUX.2 VAE переобучен так, чтобы лучше балансировать все три компонента :

```
→ улучшенная semantic alignment латентов
→ лучшая learnability
→ более высокая reconstruction fidelity
→ поддержка редактирования до 4MP
```

VAE выпущен отдельно как **open source под лицензией Apache 2.0** .

---

### Multi-reference: до 10 изображений

В FLUX.1 Kontext — один референс через sequence concatenation.

В FLUX.2 та же идея расширена до **10 референсных изображений** :

```
FLUX.1 Kontext:
[text | ref_0 | noisy target]

FLUX.2:
[text | ref_0 | ref_1 | ... | ref_9 | noisy target]
```

Каждый референс получает свой image_id в позиционном кодировании. Joint attention позволяет target токенам обращаться к любому референсу одновременно.

---

### Новые возможности управления

**JSON Structured Prompts** — работает без специального препроцессинга. JSON токенизируется как обычный текст, Mistral-3 понимает структуру нативно :

```json
{
  "subject": "woman in red dress",
  "lighting": "golden hour, side light",
  "background": "blurred city",
  "style": "editorial photography"
}
```

**HEX Color Control** — точное задание цветов:

```
"платье цвета #C41E3A"  →  точное соответствие brand guidelines
```

---

### Версии моделей FLUX.2

| Версия | Параметры | Референсы | Доступность | Лицензия |
| --- | --- | --- | --- | --- |
| FLUX.2 [pro] | н/д | до 8 | BFL Playground + API | коммерческая |
| FLUX.2 [flex] | н/д | до 10 | BFL Playground + API | коммерческая |
| FLUX.2 [dev] | 32B | до 10 | open weights, HuggingFace | некоммерческая |
| FLUX.2 [klein] | 4B / 9B | TBA | open weights (16.01.2026) | Apache 2.0 (4B) |
| FLUX.2 VAE | — | — | open source | Apache 2.0 |

FLUX.2 [dev] требует 90GB VRAM в полной конфигурации, 64GB в lowVRAM режиме. NVIDIA совместно с BFL выпустили FP8-квантизации, снижающие потребление памяти на ~40% . Для RTX 4090 используется weight streaming через ComfyUI — части модели выгружаются в системную память.

---

### Сравнение поколений

|  | FLUX.1 (2024) | FLUX.2 (ноябрь 2025) |
| --- | --- | --- |
| Параметры | 12B | 32B (dev) |
| Text encoder | T5 + CLIP | Mistral-3 (24B VLM) |
| Text embeddings | последний слой | stacked intermediate layers |
| VAE | базовый | переобученный, Apache 2.0 |
| MLP активация | GELU | SwiGLU |
| QKV + FF fusion | нет | да |
| Референсы | 1 (Kontext) | до 10 |
| Макс. разрешение | ~1MP | 4MP |
| Генерация + редактирование | разные модели | единая модель |
| Structured prompts | нет | JSON + HEX |
| Локальный запуск | RTX 3090+ | RTX 4090 (FP8 + weight streaming) |

---

> FLUX.2 — это масштабирование FLUX.1 по нескольким осям одновременно: замена текстового стека на Mistral-3 (24B VLM), переобучение VAE для решения learnability-quality-compression trilemma, расширение multi-reference до 10 изображений, и инженерные оптимизации в трансформере (SwiGLU, QKV fusion).  
> Базовая идея — rectified flow трансформер с joint attention — остаётся неизменной

## 3.6 Qwen-Image: open-source альтернатива от Alibaba

![qwen-i](assets/qwen.jpg)

Qwen-Image — семейство моделей генерации изображений от Alibaba Cloud, выпущенное в августе 2025 года . Главная причина, по которой оно заслуживает отдельного раздела: **полностью открытые веса под лицензией Apache 2.0** при качестве, сопоставимом с закрытыми системами — но только для первого поколения .

---

### Хронология выпусков

```
август 2025:    Qwen-Image           ← T2I, рендеринг текста, Apache 2.0
август 2025:    Qwen-Image-Edit      ← редактирование, Apache 2.0
сентябрь 2025:  Qwen-Image-Edit-2509 ← multi-image editing, Apache 2.0
декабрь 2025:   Qwen-Image-2512      ← реализм и детализация, Apache 2.0
декабрь 2025:   Qwen-Image-Edit-2511 ← улучшенное редактирование, Apache 2.0
февраль 2026:   Qwen-Image-2.0       ← унификация T2I+I2I, только API
```

Важное уточнение: **Qwen-Image-2.0 — не open source**. Доступен только через API Alibaba Cloud BaiLian, веса и код закрыты .

---

### Архитектура первого поколения

Qwen-Image v1 построен на **MMDiT — Multimodal Diffusion Transformer** :

```
Qwen-Image v1:
→ 20B параметров
→ 60 dual-stream MMDiT блоков
→ flow matching objective (как FLUX)
→ Apache 2.0 лицензия
→ поддержка diffusers
```

### Dual-stream: только double-stream до конца

Принципиальное архитектурное отличие от FLUX:

```
FLUX.1:
19× DoubleStream  ← раздельные веса, joint attention
      ↓
38× SingleStream  ← общие веса, объединённый поток

Qwen-Image v1:
60× MMDiT         ← раздельные веса, joint attention
                     нет SingleStream фазы
                     потоки никогда не объединяются
```

Каждый MMDiT блок — dual-stream блок с раздельными весами проекций и joint attention, аналогичный FLUX.1 DoubleStream. Отличие от FLUX: нет фазы интеграции с общими весами.

Почему это архитектурное решение имеет смысл:

```
FLUX: специализация → интеграция
→ сначала каждая модальность учится отдельно
→ потом объединяются в общий поток

Qwen-Image: только специализация
→ модальности всегда раздельны
→ взаимодействие только через joint attention
→ более предсказуемое редактирование
→ проще сохранить структуру при изменении деталей
```

### Специальные технические компоненты v1

**MSRoPE** (Multi-Scale Rotary Position Embedding) :

```
стандартный RoPE (FLUX):
→ 2D для изображений: RoPE(x, y)
→ 1D для текста: RoPE(pos)
→ раздельные схемы для каждой модальности

MSRoPE (Qwen-Image):
→ единая схема для обеих модальностей
→ изображение: RoPE(x, y) в 2D
→ текст: RoPE(pos) в 1D
→ оба в одном согласованном позиционном пространстве
```

**Нормализация :**

```
QK-Norm:   RMSNorm  ← как в FLUX, стабилизирует длинные последовательности
Остальное: LayerNorm ← в отличие от FLUX где везде RMSNorm
```

**VAE с единым энкодером и двойным декодером :**

```
энкодер:  заморожен от Wan-2.1-VAE
          ↑ стабильное латентное пространство

декодер:  два варианта
          → стандартный image decoder
          → text-rich decoder, дообученный на:
             PDF-документах, постерах,
             синтетических параграфах
             → именно это даёт качественный
               рендеринг мелкого текста
```

**Как текст взаимодействует с image latents :**

```
text features (2D tensor)
        ↓
diagonal concatenation с image latents
        ↓
joint attention над объединённой матрицей
в каждом MMDiT блоке
```

---

### Как понимаются референсы

#### В Qwen-Image (базовая, T2I)

Референсных изображений нет — только текст:

```
текст → Qwen2.5-VL → text embeddings
                           ↓
                      diagonal concat с noisy latent
                           ↓
                      MMDiT → output image
```

#### В Qwen-Image-Edit (v1, I2I)

Qwen-Image-Edit использует **два параллельных пути** для понимания одного референсного изображения :

```
одно референсное изображение
        │
        ├──→ Qwen2.5-VL (Semantic Path)
        │         ↓
        │    semantic embeddings
        │    "что это" — объекты, атрибуты,
        │    отношения, контекст
        │
        └──→ VAE Encoder (Appearance Path)
                  ↓
             appearance embeddings
             "как выглядит" — точные цвета,
             текстуры, пиксельные детали
```

Оба представления объединяются внутри MMDiT через **cross-attention**:

```
noisy target latent  → Q
semantic embeddings  ┐
appearance embeddings├→ конкатенируются → K, V
text embeddings      ┘
        ↓
cross-attention(Q, K, V):
каждый target токен смотрит на все три источника
и сам решает что взять из каждого
```

Зачем два пути из одного изображения:

```
только Semantic (VLM):
→ понимает "что" сохранить
→ теряет точные пиксельные детали
→ лицо меняется, цвета плывут

только Appearance (VAE):
→ помнит пиксельную структуру
→ не понимает семантику
→ не может правильно изменить контекст

вместе:
→ semantic: "что это и что нужно сохранить"
→ appearance: "как именно это выглядит пиксельно"
```

Это принципиально отличается от FLUX Kontext:

```
FLUX Kontext:
ref tokens конкатенируются в одну последовательность
→ self-attention, все токены равноправны
→ модель сама разбирается через 3D RoPE

Qwen-Image-Edit:
явное разделение ролей через cross-attention
target latent → Q  (что генерируем)
условия       → K, V (на что смотрим)
→ структурно закреплено: target управляет,
  условия предоставляют информацию
```

---

### Ключевая особенность v1: рендеринг текста

Главное достижение первого поколения — **state-of-the-art рендеринг текста**, особенно китайского :

```
проблема предыдущих моделей:
→ FLUX, SD — текст на изображениях нечитаем
→ символы искажаются или смешиваются
→ иероглифы практически невозможны

Qwen-Image:
→ единственная open-source модель
  с качественным рендерингом китайского текста 
→ текст интегрируется в изображение
  как часть композиции
→ поддержка иероглифов, пунктуации, layout
```

Достигается двумя механизмами одновременно:

```
1. text-rich VAE decoder
   → дообучен на PDF, постерах, синтетических текстах
   → умеет реконструировать мелкие символы

2. Qwen2.5-VL как семантический поток
   → нативно понимает китайский язык
   → передаёт точную семантику символов в MMDiT
```

---

### Что нового в Qwen-Image-2512 (декабрь 2025)

```
было (август 2025):
→ "AI look" у людей — пластиковые лица
→ недостаточная детализация текстур
→ ошибки в компоновке текста на изображении

стало (декабрь 2025):
→ реалистичные люди: детали лица, возраст, кожа
→ тонкие природные текстуры: вода, мех, материалы
→ улучшенный layout при рендеринге текста
```

По результатам 10 000+ слепых тестов на AI Arena — сильнейшая open-source модель на момент выхода .

---

### Архитектура второго поколения: Qwen-Image-2.0 (февраль 2026)

Qwen-Image-2.0 — кардинальный архитектурный пересмотр, но **закрытая модель** :

```
v1 (единый MMDiT 20B, open source):
→ одна модель делает всё
→ отдельные модели для T2I и I2I

v2 (~15B = 8B + 7B, только API):
→ специализация: VLM понимает, DiT генерирует
→ единая модель для T2I и I2I
→ один forward pass
```

Компоненты :

```
Qwen-Image-2.0:

Qwen3-VL (8B)   ← condition encoder
                   нативно понимает текст и изображения
      ↓
condition embeddings
      ↓
MMDiT (7B)      ← diffusion decoder
      ↓
output image
```

---

### Версии и доступность

| Версия | Параметры | Дата | Лицензия | Доступность |
| --- | --- | --- | --- | --- |
| Qwen-Image | 20B | авг 2025 | Apache 2.0 | HuggingFace, ModelScope |
| Qwen-Image-Edit | 20B | авг 2025 | Apache 2.0 | HuggingFace, ModelScope |
| Qwen-Image-Edit-2509 | 20B | сен 2025 | Apache 2.0 | HuggingFace, ModelScope |
| Qwen-Image-2512 | 20B | дек 2025 | Apache 2.0 | HuggingFace, ModelScope |
| Qwen-Image-Edit-2511 | 20B | дек 2025 | Apache 2.0 | HuggingFace, ModelScope |
| Qwen-Image-2.0 | ~15B (8B+7B) | фев 2026 | закрытая | только API |

---

### Сравнение с FLUX

|  | FLUX.1 | Qwen-Image v1 | Qwen-Image-2.0 |
| --- | --- | --- | --- |
| Параметры | 12B | 20B | ~15B (8B+7B) |
| Архитектура | 19× DoubleStream + 38× SingleStream | 60× MMDiT dual-stream | Qwen3-VL + MMDiT |
| Text encoder | T5 + CLIP | Qwen2.5-VL (встроен) | Qwen3-VL (8B, отдельный) |
| Референс механизм | sequence concat + 3D RoPE | dual-path (semantic + appearance) + cross-attention | неизвестно (код закрыт) |
| Позиционное кодирование | 2D RoPE / 3D RoPE (Kontext) | MSRoPE | неизвестно |
| Нормализация | RMSNorm везде | QK: RMSNorm, остальное: LayerNorm | неизвестно |
| SingleStream фаза | да (38 блоков) | нет | нет |
| Рендеринг текста | слабый | SOTA, особенно Chinese | профессиональный |
| Редактирование | отдельная модель (Kontext) | отдельная модель | единая модель |
| Разрешение | ~1MP | гибкое | native 2K |
| Open source | dev — некоммерческая | Apache 2.0 | нет, только API |
